# 1- Importing Libraries

In [1]:
! pip install pandas
! pip install numpy
! pip install scikit-learn
! pip install nltk
! pip install seaborn
! pip install imbalanced-learn
! pip install langchain
! pip install langchain-huggingface
! pip install sentence-transformers
! pip install xgboost
! pip install python-dotenv

In [2]:
import pandas as pd
import numpy as np
import re
from tqdm import tqdm
import nltk
import seaborn as sns
from dotenv import load_dotenv
from sklearn.model_selection import train_test_split
from sklearn.svm import SVC
from sklearn.naive_bayes import MultinomialNB
from nltk.tokenize import sent_tokenize,word_tokenize
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from imblearn.over_sampling import SMOTE
from xgboost import XGBClassifier
from nltk.stem import WordNetLemmatizer
from nltk.corpus import stopwords
from sklearn.feature_extraction.text import TfidfVectorizer,CountVectorizer
from imblearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score,f1_score,classification_report,confusion_matrix
from langchain_huggingface import HuggingFaceEmbeddings
nltk.download("stopwords")
nltk.download("punkt_tab")
lemmatizer = WordNetLemmatizer()
nltk.download('wordnet')
load_dotenv()

/Users/prabhsandhu/Downloads/Machine_Learning_Project/venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
[nltk_data] Downloading package stopwords to
[nltk_data]     /Users/prabhsandhu/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     /Users/prabhsandhu/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package wordnet to
[nltk_data]     /Users/prabhsandhu/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


False

# 2- Loading Dataset

In [4]:
df = pd.read_csv("Reviews.csv")
df = df.sample(25000)

# 3- Understanding Dataset

In [8]:
df.columns

Index(['Id', 'ProductId', 'UserId', 'ProfileName', 'HelpfulnessNumerator',
       'HelpfulnessDenominator', 'Score', 'Time', 'Summary', 'Text'],
      dtype='str')

In [31]:
df.shape

(200000, 10)

In [ ]:
df.nunique()
df["Score"].unique()

array([5, 1, 4, 2, 3])

In [ ]:
df['HelpfulnessNumerator']

0         1
1         0
2         1
3         3
4         0
         ..
568449    0
568450    0
568451    2
568452    1
568453    0
Name: HelpfulnessNumerator, Length: 568454, dtype: int64

# 4- Preprocessing the Dataset

In [4]:
df.isnull().sum()

Id                        0
ProductId                 0
UserId                    0
ProfileName               4
HelpfulnessNumerator      0
HelpfulnessDenominator    0
Score                     0
Time                      0
Summary                   0
Text                      0
dtype: int64

In [4]:
df.dropna(inplace=True)

In [5]:
df.duplicated().sum()

np.int64(0)

In [5]:
df.drop(['Id','UserId','Time','Summary','HelpfulnessNumerator','HelpfulnessDenominator','ProfileName'],axis=1,inplace=True)

In [6]:
df["Sentiment"] = df["Score"].apply(lambda x:"Negative" if x<=2 else ("Neutral" if x==3 else "Positive"))

In [7]:
df.drop("Score",axis=1,inplace=True)

# 5- Text Preprocessing Using NLP

In [8]:
lemmatizer = WordNetLemmatizer()
stop_words = set(stopwords.words("english"))  # load ONCE

text_preprocessed = []

for text in tqdm(df["Text"].astype(str)):  # ensure all rows are strings
    text = text.lower()
    text = re.sub(r'[^A-Za-z0-9]', ' ', text)
    
    words = word_tokenize(text)
    words = [lemmatizer.lemmatize(word) for word in words if word not in stop_words and len(word) >= 2]
    
    text_preprocessed.append(" ".join(words))

100%|██████████| 25000/25000 [00:10<00:00, 2326.91it/s]


# 6- Training Machine Learning Models Using LLM Embeddings as Technique

In [9]:
df

,Id,ProductId,UserId,ProfileName,HelpfulnessNumerator,HelpfulnessDenominator,Score,Time,Summary,Text
120419,120420,B005K4Q37A,A2G5994VXZSPLI,TLB,0,0,5,1340496000,These taste so good!,"I love this coffee. It is wonderful, smoothe a..."
392099,392100,B0013ES5WC,AWNOWP4QGRXK3,"P. Ju ""Scootrat""",0,3,4,1247616000,It filters,"It does its job. Pretty expensive, at $3-$4 a ..."
392059,392060,B001RV8CHY,A154TLVB3GAIU7,Diane R. Catanzaro,0,0,4,1348185600,Excellent on fish!,"Excellent for grilling, frying, or baking fish..."
416153,416154,B002HFWMOI,AJ3QAM3XX4BUE,"Barefootokie75 ""Sarah Varadi Miller""",0,0,5,1306713600,Happy,I've been using Nestle's Fat Free Hot Cocoa Mi...
369143,369144,B000FNEZGM,A3KQUWSBYUQGOI,J. Trellu,0,0,5,1219968000,Our daughter LOVES these cookie bars!,"Our daughter has several food allergies, so Na..."
...,...,...,...,...,...,...,...,...,...,...
65187,65188,B004U8WODY,A30J8KM3H9TDBU,"Clasina Gm Oneill ""Maria""",0,0,4,1333584000,Lovely with tea.,"These are good digestives, crunchy, not too sw..."
517024,517025,B000G6L2ZU,A2T2MJT8LFLBXH,"Clow Angel ""my thoughts, my memories.""",2,2,5,1184976000,"Cheap, Well Worth It.","Now, I've always been a fan of gummies. I love..."
28396,28397,B007OXJMD2,A30GEP1I3ZJABG,J. Thoman,0,0,4,1326153600,"Good, but a little pricey","I enjoy a cup of hazelnut coffee on occasion, ..."
247697,247698,B001E6GFZI,AZQ05PT95RA5H,W. Mattia,4,5,5,1196294400,"Must have, no joke",I do not think I can go a day without it. It ...


In [9]:
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from imblearn.pipeline import Pipeline
from xgboost import XGBClassifier
from langchain_huggingface import HuggingFaceEmbeddings
X = text_preprocessed
y = df["Sentiment"]

le = LabelEncoder()
y_encoded = le.fit_transform(y)
model = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

def batch_embed(texts, batch_size=64):
    embeddings = []
    for i in range(0, len(texts), batch_size):
        batch = texts[i:i+batch_size]
        emb = model.embed_documents(batch)
        embeddings.extend(emb)
        print(f"Processed {i+len(batch)}/{len(texts)}")
    return np.array(embeddings)

import os

if os.path.exists("embeddings.npy"):
    print("Loading cached embeddings...")
    X_embeddings = np.load("embeddings.npy")
else:
    print("Generating embeddings (one-time, slow)...")
    X_embeddings = batch_embed(X)
    np.save("embeddings.npy", X_embeddings)
    print("Embeddings saved!")

X_train, X_test, y_train, y_test = train_test_split(
    X_embeddings, y_encoded, test_size=0.3, random_state=42
)

pipe = Pipeline([
    ('xgb', XGBClassifier(
        n_estimators=100,          # 🔥 fast
        max_depth=5,
        learning_rate=0.15,
        subsample=0.8,
        colsample_bytree=0.8,
        objective='multi:softprob',
        num_class=3,
        eval_metric='mlogloss',
        n_jobs=-1                 # use all CPU cores
    ))
])

pipe.fit(X_train, y_train)

y_pred = pipe.predict(X_test)

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 4450.53it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Generating embeddings (one-time, slow)...
Processed 64/25000
Processed 128/25000
Processed 192/25000
Processed 256/25000
Processed 320/25000
Processed 384/25000
Processed 448/25000
Processed 512/25000
Processed 576/25000
Processed 640/25000
Processed 704/25000
Processed 768/25000
Processed 832/25000
Processed 896/25000
Processed 960/25000
Processed 1024/25000
Processed 1088/25000
Processed 1152/25000
Processed 1216/25000
Processed 1280/25000
Processed 1344/25000
Processed 1408/25000
Processed 1472/25000
Processed 1536/25000
Processed 1600/25000
Processed 1664/25000
Processed 1728/25000
Processed 1792/25000
Processed 1856/25000
Processed 1920/25000
Processed 1984/25000
Processed 2048/25000
Processed 2112/25000
Processed 2176/25000
Processed 2240/25000
Processed 2304/25000
Processed 2368/25000
Processed 2432/25000
Processed 2496/25000
Processed 2560/25000
Processed 2624/25000
Processed 2688/25000
Processed 2752/25000
Processed 2816/25000
Processed 2880/25000
Processed 2944/25000
Processe

In [19]:
print(accuracy_score(y_test,y_pred)*100)
print(f1_score(y_test,y_pred,average="weighted")*100)
print(classification_report(y_test,y_pred))
print(confusion_matrix(y_test,y_pred))

82.73333333333333
78.68558913080726
              precision    recall  f1-score   support

           0       0.69      0.38      0.49      1056
           1       0.70      0.05      0.10       542
           2       0.84      0.98      0.90      5902

    accuracy                           0.83      7500
   macro avg       0.74      0.47      0.50      7500
weighted avg       0.81      0.83      0.79      7500

[[ 396    4  656]
 [  65   28  449]
 [ 113    8 5781]]
